In [ ]:
##Configuration
ASTRA_DB_API_ENDPOINT = "https://cbaf9e9e-2dba-4f28-b9d7-1df88633f175-us-east-2.apps.astra.datastax.com"
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")

In [8]:

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7485.42it/s]


In [5]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [9]:
from langchain_astradb import AstraDBVectorStore
vector_store = AstraDBVectorStore(
    embedding = embeddings,
    api_endpoint= ASTRA_DB_API_ENDPOINT,
    collection_name = "astra_vector_langchain",
    token = ASTRA_DB_APPLICATION_TOKEN,
    namespace = None
)

In [10]:
vector_store

In [12]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [13]:
vector_store.add_documents(documents= documents)


['3dd6d40097884f2e89f4f1e52793cee5',
 'b957ac8620e24246838d7ac112eb3327',
 '01bbf5ce87c74d179b59a2aa7b9d6d70',
 '7359566db8ad4caeb470cb4b2973cb32',
 '113a6893268b467fa3cd68554937b6a4',
 '5e39365829a84bd2a06e86a8ea6bb893',
 'ba3b2ddf065d41d59d51ac07d2a72999',
 '9233f5d02844488683ac541366d1f364',
 '5b8ead62d9624f5a8d89eaeff783c56c',
 '3210c03bd2a3468a9d713d176f77c770']

In [14]:
vector_store

In [16]:
result = vector_store.similarity_search("Langchain provides bastraction to make working with LLMs easy.",
                               k=3,
                               filter = {"source": "tweet"})
for res in result:
    print(f'*"{res.page_content}",metadata : {res.metadata}')

*"Building an exciting new project with LangChain - come check it out!",metadata : {'source': 'tweet'}
*"LangGraph is the best framework for building stateful, agentic applications!",metadata : {'source': 'tweet'}
*"I have a bad feeling I am going to get deleted :(",metadata : {'source': 'tweet'}


In [18]:
results = vector_store.similarity_search_with_score("Langchain provides bastraction to make working with LLMs easy.",k=3, filter = {"source":"tweet"})
for res,score in results:
    print(f'* [SIM] = {score:.2f}] "{res.page_content}", metadata = {res.metadata}')

* [SIM] = 0.79] "Building an exciting new project with LangChain - come check it out!", metadata = {'source': 'tweet'}
* [SIM] = 0.72] "LangGraph is the best framework for building stateful, agentic applications!", metadata = {'source': 'tweet'}
* [SIM] = 0.51] "I have a bad feeling I am going to get deleted :(", metadata = {'source': 'tweet'}


In [19]:
##Retriever:
retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs = {'k':1, 'score_threshold':0.5}
)
retriever.invoke("Stealing from bank is a crime",filter = {'source':'news'})

[Document(id='7359566db8ad4caeb470cb4b2973cb32', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [20]:
retriever = vector_store.as_retriever(
    search_type = "mmr",
    search_kwargs = {'k':1}
)
retriever.invoke("Stealing from bank is a crime", filter = {'source':'news'})

[Document(id='7359566db8ad4caeb470cb4b2973cb32', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]